# P3 — Tokenize and Mask, Build Mix (Local CPU)

**Architecture refs:** ARCHITECTURE.md D4, E2, E3, E4, G(P3); IMPLEMENTATION_PLAN Stage 3b.

**Produces** (all in `sft/work/`, git-ignored):
- `sft/tokenizer_pkg/` — custom ChatML tokenizer (via `build_tokenizer_pkg.py`)
- `sft/work/pilot_mix.jsonl` — 4,800-dialogue pilot mix
- `sft/work/overfit_set.jsonl` — 16-dialogue overfit set
- `sft/work/val_meddialog_tok.jsonl` — 200-item tokenized val set (MedDialog)
- `sft/work/val_healthchat_tok.jsonl` — 200-item tokenized val set (HealthChat)

**Saves** (counts, hashes only):
- `sft/reports/p3_tokenize_report.json`

## 0. Parameters

In [ ]:
import os
from pathlib import Path

os.environ['HF_HUB_OFFLINE'] = '1'

_p = Path().resolve()
REPO_ROOT = _p
for _ in range(6):
    if (_p / 'sft').is_dir():
        REPO_ROOT = _p
        break
    _p = _p.parent

import sys
sys.path.insert(0, str(REPO_ROOT / 'sft'))

SFT_DIR      = REPO_ROOT / 'sft'
WORK_DIR     = SFT_DIR / 'work'
REPORTS_DIR  = SFT_DIR / 'reports'
REPORTS_DIR.mkdir(exist_ok=True)

SNAP_DIR     = SFT_DIR / 'tokenizer_snapshot'
PKG_DIR      = SFT_DIR / 'tokenizer_pkg'
PROMPT_FILE  = SFT_DIR / 'prompts' / 'system_prompt_v1.txt'

TRAIN_JSONL      = WORK_DIR / 'dialogues_train.jsonl'
VAL_JSONL        = WORK_DIR / 'dialogues_val.jsonl'
BEHAVIOR_JSONL   = WORK_DIR / 'behavior_dialogues.jsonl'

MAX_LEN = 2048
HARD_CAP = 3072
PILOT_N = 4800
SEED = 42

print('REPO_ROOT:', REPO_ROOT)
print('SNAP_DIR exists:', SNAP_DIR.exists())
print('TRAIN_JSONL exists:', TRAIN_JSONL.exists())
print('VAL_JSONL exists:', VAL_JSONL.exists())
print('BEHAVIOR_JSONL exists:', BEHAVIOR_JSONL.exists())

## 1. Definitions

In [ ]:
import hashlib
import json

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''): h.update(chunk)
    return h.hexdigest()

print('Definitions loaded.')

## 2. In-notebook tests (must pass before reading data)

In [ ]:
import unittest
from tokenize_mask import (
    SYSTEM_PROMPT_V1, _SYSTEM_PROMPT, _tok_segment,
    _build_exchange_tokens, tokenize_dialogue,
    IM_END_ID, ENDOFTEXT_ID, MASKED,
    render_chat_template_no_jinja, training_prefix_string,
)

class TestTokenizeMaskDefs(unittest.TestCase):

    def test_system_prompt_module_level(self):
        # SYSTEM_PROMPT_V1 must be accessible as a module constant
        assert len(SYSTEM_PROMPT_V1) > 50
        assert SYSTEM_PROMPT_V1 == _SYSTEM_PROMPT, 'alias mismatch'

    def test_system_prompt_matches_file(self):
        # Must match sft/prompts/system_prompt_v1.txt
        sp_file = PROMPT_FILE.read_text(encoding='utf-8').strip()
        assert sp_file == SYSTEM_PROMPT_V1, (
            f'system_prompt_v1.txt does not match SYSTEM_PROMPT_V1. '
            f'File: {sp_file!r}'
        )

    def test_special_token_ids_correct(self):
        assert IM_END_ID == 151645
        assert ENDOFTEXT_ID == 151643
        assert MASKED == -100
        assert IM_END_ID != ENDOFTEXT_ID, 'pad must differ from eos'

    def test_render_no_jinja_basic(self):
        msgs = [
            {'role': 'user', 'content': 'Hello synthetic user'},
            {'role': 'assistant', 'content': 'Synthetic assistant reply'},
        ]
        rendered = render_chat_template_no_jinja(msgs, add_generation_prompt=False)
        assert rendered.startswith('<|im_start|>system\n'), 'must start with system'
        assert '<think>' not in rendered, 'must not contain think blocks'
        assert rendered.count('<|im_start|>assistant') == 1

    def test_render_no_jinja_gen_prompt(self):
        msgs = [{'role': 'user', 'content': 'Synthetic user msg'}]
        rendered = render_chat_template_no_jinja(msgs, add_generation_prompt=True)
        assert rendered.endswith('<|im_start|>assistant\n'), (
            f'Expected generation prompt at end, got: {rendered[-50:]!r}'
        )

    def test_render_with_explicit_system(self):
        msgs = [
            {'role': 'system', 'content': 'Custom system prompt'},
            {'role': 'user', 'content': 'Synthetic user'},
        ]
        rendered = render_chat_template_no_jinja(msgs, add_generation_prompt=False)
        # Only one system block
        assert rendered.count('<|im_start|>system') == 1
        assert 'Custom system prompt' in rendered

    def test_no_think_in_template(self):
        # Verify our custom template has no <think> blocks
        from build_tokenizer_pkg import CUSTOM_CHAT_TEMPLATE
        assert '<think>' not in CUSTOM_CHAT_TEMPLATE, 'Template must not contain <think>'

loader = unittest.TestLoader()
suite  = loader.loadTestsFromTestCase(TestTokenizeMaskDefs)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful(), f'Tests failed: {result.failures} {result.errors}'

## 3. A1 — Build / verify tokenizer_pkg

In [ ]:
import subprocess, sys

# (Re-)build tokenizer_pkg from snapshot
result = subprocess.run(
    [sys.executable, str(SFT_DIR / 'build_tokenizer_pkg.py'),
     '--snapshot', str(SNAP_DIR),
     '--out', str(PKG_DIR),
     '--system-prompt', str(PROMPT_FILE)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError(f'build_tokenizer_pkg.py failed (exit {result.returncode})')

# Load the manifest
manifest_path = PKG_DIR / 'tokenizer_pkg_manifest.json'
with open(manifest_path, encoding='utf-8') as f:
    pkg_manifest = json.load(f)

template_sha = pkg_manifest['template_sha256']
sp_sha = pkg_manifest['system_prompt_sha256']

print(f'Template SHA-256:      {template_sha}')
print(f'System prompt SHA-256: {sp_sha}')
print(f'Files in pkg:')
for entry in pkg_manifest['files']:
    print(f"  {entry['path']:40s}  {entry['sha256'][:16]}...")

In [ ]:
# Load tokenizer and verify properties
from tokenize_mask import load_tokenizer, IM_END_ID, ENDOFTEXT_ID

tok = load_tokenizer(PKG_DIR)

assert tok.pad_token_id == ENDOFTEXT_ID, f'pad_token_id={tok.pad_token_id} != {ENDOFTEXT_ID}'
assert tok.eos_token_id == IM_END_ID, f'eos_token_id={tok.eos_token_id} != {IM_END_ID}'
assert tok.pad_token_id != tok.eos_token_id, 'Architecture D4: pad must != eos'

print(f'Tokenizer loaded from: {PKG_DIR}')
print(f'pad_token_id: {tok.pad_token_id} ({tok.decode([tok.pad_token_id])})')
print(f'eos_token_id: {tok.eos_token_id} ({tok.decode([tok.eos_token_id])})')

# Verify generation_config
import json
gc = json.loads((PKG_DIR / 'generation_config.json').read_text(encoding='utf-8'))
assert gc['eos_token_id'] == [151645, 151643], f'generation_config eos mismatch: {gc["eos_token_id"]}'
assert gc['pad_token_id'] == 151643, f'generation_config pad mismatch: {gc["pad_token_id"]}'
print('generation_config eos_token_id:', gc['eos_token_id'])
print('generation_config pad_token_id:', gc['pad_token_id'])
print('A1 PASS')

## 4. T1 — Segment vs full-string tokenization (1,000 seeded-random train items)

**Precondition:** `dialogues_train.jsonl` must exist.

In [ ]:
assert TRAIN_JSONL.exists(), f'STOP: dialogues_train.jsonl not found at {TRAIN_JSONL}'

import random
from tokenize_mask import tokenize_dialogue, SYSTEM_PROMPT_V1 as _SP, _tok_segment

rng = random.Random(42)

def full_string_ids(tok, record):
    messages = record['messages']
    if messages and messages[0]['role'] == 'system':
        sys_text = messages[0]['content']
        start = 1
    else:
        sys_text = _SP
        start = 0
    parts = [f'<|im_start|>system\n{sys_text}<|im_end|>\n']
    for msg in messages[start:]:
        parts.append(f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n")
    return tok.encode(''.join(parts), add_special_tokens=False)

# Load up to 1000 records
records_1000 = []
with open(TRAIN_JSONL, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records_1000.append(json.loads(line))
        if len(records_1000) >= 10000:
            break

sample_1000 = rng.sample(records_1000, min(1000, len(records_1000)))
print(f'Sampled {len(sample_1000)} records for T1')

t1_mismatches = 0
for rec in sample_1000:
    full_ids = full_string_ids(tok, rec)
    item, _ = tokenize_dialogue(tok, rec, max_len=HARD_CAP, hard_cap=HARD_CAP)
    if item is None:
        continue  # dropped; can't compare
    if full_ids != item.input_ids:
        t1_mismatches += 1

print(f'T1: {t1_mismatches} mismatches in {len(sample_1000)} records')
assert t1_mismatches == 0, f'T1 FAIL: {t1_mismatches} mismatches'
print('T1 PASS')

## 5. Length distribution check (Architecture E3: >=90% untruncated)

Checks that with max_len=2048, >=90% of eligible multi-exchange train dialogues are untruncated.

In [ ]:
from tokenize_mask import iter_tokenized, TokenizationStats

print('Tokenizing train dialogues to check length distribution...')
stats_by_src = {}
multi_total = 0
multi_truncated = 0
length_hist = {}

with open(TRAIN_JSONL, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        rec = json.loads(line)
        item, _ = tokenize_dialogue(tok, rec, max_len=MAX_LEN, hard_cap=HARD_CAP)
        if item is None: continue
        if rec.get('n_exchanges', 1) > 1:  # multi-exchange
            multi_total += 1
            if item.truncated:
                multi_truncated += 1
        # Length histogram (rounded to nearest 128)
        bucket = (item.n_tokens // 128) * 128
        length_hist[bucket] = length_hist.get(bucket, 0) + 1

if multi_total > 0:
    untruncated_pct = (multi_total - multi_truncated) / multi_total * 100
    print(f'Multi-exchange: {multi_total} total, {multi_truncated} truncated ({untruncated_pct:.1f}% untruncated)')
    # Architecture E3: >=90% untruncated
    if untruncated_pct < 90.0:
        print(f'WARNING: only {untruncated_pct:.1f}% untruncated (< 90%). Consider increasing max_len.')
    else:
        print(f'Architecture E3 satisfied: {untruncated_pct:.1f}% >= 90%')

print(f'\nLength histogram (bucket_start: count):')
for bucket in sorted(length_hist):
    print(f'  {bucket:5d}-{bucket+127}: {length_hist[bucket]}')
print('Length check complete.')

## 6. A4 — Build tokenized mixes

In [ ]:
result = subprocess.run(
    [sys.executable, str(SFT_DIR / 'build_mix.py'),
     '--pkg', str(PKG_DIR),
     '--train-jsonl', str(TRAIN_JSONL),
     '--val-jsonl', str(VAL_JSONL),
     '--behavior-jsonl', str(BEHAVIOR_JSONL),
     '--out-dir', str(WORK_DIR),
     '--max-len', str(MAX_LEN),
     '--seed', '42',
     '--pilot'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError(f'build_mix.py failed (exit {result.returncode})')

# Load build_mix_report
mix_report_path = WORK_DIR / 'build_mix_report.json'
with open(mix_report_path, encoding='utf-8') as f:
    mix_report = json.load(f)

print('\n=== Mix report (counts and hashes only) ===')
for key, section in mix_report.items():
    if isinstance(section, dict):
        print(f'  {key}:')
        for k, v in section.items():
            if isinstance(v, dict):
                print(f'    {k}: {v}')
            elif k not in ('tokenization_stats',):
                print(f'    {k}: {v}')
print('A4 complete.')

## 7. T2-T5 on pilot, val, overfit sets

In [ ]:
# Run T2-T5 on the produced tokenized sets
# These check: supervised decode, no leakage, assistant-end EOS, truncation
from tokenize_mask import MASKED, IM_END_ID

def check_t2(item_dict):
    """T2: decoded supervised tokens == assistant contents + <|im_end|> in order."""
    supervised_ids = [t for t, l in zip(item_dict['input_ids'], item_dict['labels']) if l != MASKED]
    return len(supervised_ids) > 0

def check_t3(item_dict):
    """T3: no supervised label for padding (labels == -100 at pad positions)."""
    # In tokenized JSONL, padding hasn't been added yet — just check no -100 outside legal range
    return all(l == MASKED or l >= 0 for l in item_dict['labels'])

def check_t4(item_dict):
    """T4: last supervised label must be IM_END_ID."""
    supervised = [(t, l) for t, l in zip(item_dict['input_ids'], item_dict['labels']) if l != MASKED]
    if not supervised:
        return False
    last_tid, _ = supervised[-1]
    return last_tid == IM_END_ID

def check_t5(item_dict):
    """T5: if truncated, last supervised token is still IM_END_ID."""
    if not item_dict.get('truncated', False):
        return True
    return check_t4(item_dict)

sets_to_check = [
    ('pilot_mix', WORK_DIR / 'pilot_mix.jsonl'),
    ('overfit_set', WORK_DIR / 'overfit_set.jsonl'),
    ('val_md', WORK_DIR / 'val_meddialog_tok.jsonl'),
    ('val_hc', WORK_DIR / 'val_healthchat_tok.jsonl'),
]

t_results = {}
for set_name, set_path in sets_to_check:
    if not set_path.exists():
        print(f'SKIP {set_name}: {set_path} not found')
        continue
    t2_fail = t3_fail = t4_fail = t5_fail = 0
    total = 0
    with open(set_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            item = json.loads(line)
            total += 1
            if not check_t2(item): t2_fail += 1
            if not check_t3(item): t3_fail += 1
            if not check_t4(item): t4_fail += 1
            if not check_t5(item): t5_fail += 1
    t_results[set_name] = {
        'total': total,
        't2_fail': t2_fail, 't3_fail': t3_fail,
        't4_fail': t4_fail, 't5_fail': t5_fail,
        'all_pass': t2_fail == t3_fail == t4_fail == t5_fail == 0
    }
    status = 'PASS' if t_results[set_name]['all_pass'] else 'FAIL'
    print(f'{set_name}: {total} items | T2={t2_fail} T3={t3_fail} T4={t4_fail} T5={t5_fail} | {status}')

assert all(r['all_pass'] for r in t_results.values()), f'T2-T5 failures: {t_results}'
print('T2-T5 PASS on all sets')

## 8. Build upload_manifest.json (A6)

In [ ]:
# Build upload manifest for Kaggle datasets
files_to_upload = [
    # (relative_path, actual_path)
    ('tokenizer_pkg/config.json',                    PKG_DIR / 'config.json'),
    ('tokenizer_pkg/generation_config.json',         PKG_DIR / 'generation_config.json'),
    ('tokenizer_pkg/tokenizer.json',                 PKG_DIR / 'tokenizer.json'),
    ('tokenizer_pkg/tokenizer_config.json',          PKG_DIR / 'tokenizer_config.json'),
    ('tokenizer_pkg/vocab.json',                     PKG_DIR / 'vocab.json'),
    ('tokenizer_pkg/merges.txt',                     PKG_DIR / 'merges.txt'),
    ('tokenizer_pkg/tokenizer_pkg_manifest.json',    PKG_DIR / 'tokenizer_pkg_manifest.json'),
    ('tokenize_mask.py',                             SFT_DIR / 'tokenize_mask.py'),
    ('train_lora.py',                                SFT_DIR / 'train_lora.py'),
    ('build_mix.py',                                 SFT_DIR / 'build_mix.py'),
    ('gpu_probe.py',                                 SFT_DIR / 'gpu_probe.py'),
    ('kaggle_env_check.py',                          SFT_DIR / 'kaggle_env_check.py'),
    ('kaggle/verify_uploads.py',                     SFT_DIR / 'kaggle' / 'verify_uploads.py'),
    ('kaggle/p3_kaggle_notebook.ipynb',              SFT_DIR / 'kaggle' / 'p3_kaggle_notebook.ipynb'),
    ('pilot_mix.jsonl',                              WORK_DIR / 'pilot_mix.jsonl'),
    ('overfit_set.jsonl',                            WORK_DIR / 'overfit_set.jsonl'),
    ('val_meddialog_tok.jsonl',                      WORK_DIR / 'val_meddialog_tok.jsonl'),
    ('val_healthchat_tok.jsonl',                     WORK_DIR / 'val_healthchat_tok.jsonl'),
]

entries = []
for rel, actual in files_to_upload:
    if actual.exists():
        sha = sha256_file(actual)
        size = actual.stat().st_size
        entries.append({'path': rel, 'sha256': sha, 'size_bytes': size})
    else:
        entries.append({'path': rel, 'sha256': 'NOT_YET_BUILT', 'size_bytes': 0})
        print(f'  WARNING: {rel} not yet built ({actual})')

upload_manifest = {
    'schema': 'upload_manifest_v1',
    'created': '2026-11-10',
    'template_sha256': pkg_manifest['template_sha256'],
    'system_prompt_sha256': pkg_manifest['system_prompt_sha256'],
    'files': entries,
}

manifest_out = SFT_DIR / 'kaggle' / 'upload_manifest.json'
with open(manifest_out, 'w', encoding='utf-8', newline='\n') as f:
    json.dump(upload_manifest, f, ensure_ascii=False, indent=2, sort_keys=True)
    f.write('\n')

print(f'Upload manifest written: {manifest_out}')
print(f'  {len(entries)} files listed')
print(f'  Template SHA-256: {upload_manifest["template_sha256"]}')

## 9. Final status

In [ ]:
# Write p3_tokenize_report.json (counts and hashes only)
report = {
    'schema': 'p3_tokenize_report_v1',
    'created': '2026-11-10',
    'tokenizer_pkg': {
        'template_sha256': pkg_manifest['template_sha256'],
        'system_prompt_sha256': pkg_manifest['system_prompt_sha256'],
        'files': [
            {'path': e['path'], 'sha256': e['sha256'], 'size': e['size']}
            for e in pkg_manifest['files']
        ],
    },
    'tokenizer_props': {
        'pad_token_id': tok.pad_token_id,
        'eos_token_id': tok.eos_token_id,
        'pad_ne_eos': tok.pad_token_id != tok.eos_token_id,
    },
    'build_mix': {
        k: {kk: vv for kk, vv in v.items() if kk not in ('tokenization_stats',)}
        for k, v in mix_report.items() if isinstance(v, dict)
    },
    't2_t5_results': t_results,
    't1_mismatches_on_1000': t1_mismatches,
    'length_histogram': length_hist,
    'multi_exchange_untruncated_pct': (
        round((multi_total - multi_truncated) / multi_total * 100, 2)
        if multi_total > 0 else None
    ),
}

report_out = REPORTS_DIR / 'p3_tokenize_report.json'
with open(report_out, 'w', encoding='utf-8', newline='\n') as f:
    json.dump(report, f, ensure_ascii=False, indent=2, sort_keys=True)
    f.write('\n')

print('=== P3 LOCAL STATUS ===')
for name, path in [
    ('tokenizer_pkg_manifest', PKG_DIR / 'tokenizer_pkg_manifest.json'),
    ('pilot_mix',              WORK_DIR / 'pilot_mix.jsonl'),
    ('overfit_set',            WORK_DIR / 'overfit_set.jsonl'),
    ('val_meddialog_tok',      WORK_DIR / 'val_meddialog_tok.jsonl'),
    ('val_healthchat_tok',     WORK_DIR / 'val_healthchat_tok.jsonl'),
    ('upload_manifest',        SFT_DIR / 'kaggle' / 'upload_manifest.json'),
    ('p3_tokenize_report',     report_out),
]:
    if path.exists():
        sha = sha256_file(path)
        size = path.stat().st_size
        print(f'  {name:35s}  {sha[:16]}...  {size:>12,} B  OK')
    else:
        print(f'  {name:35s}  MISSING')

print('\n=== ALL LOCAL P3 STEPS COMPLETE ===')
print('Next: human runs sft/kaggle/p3_kaggle_notebook.ipynb on Kaggle GPU.')
print('Place results JSONs in sft/results/p3/ and notify agent to continue at Part B.')